# Cox 基线模型（仅建模与预测）
本笔记本只负责训练 Cox 模型并输出预测结果，供评分笔记本使用。

In [6]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

from sksurv.linear_model import CoxPHSurvivalAnalysis
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 2. 数据加载与预处理
读取清洗数据，生成生存分析标签并划分训练/验证集。

In [7]:
df_train = pd.read_csv("data_clean/train_clean.csv")

# 目标列
if "time_to_hit_hours" not in df_train.columns or "event" not in df_train.columns:
    raise ValueError("缺少必要列: time_to_hit_hours 或 event")

# 准备 y

def prepare_y(event_series, duration_series):
    return np.array(
        list(zip(event_series.astype(bool), duration_series)),
        dtype=[("event", "?"), ("duration", "<f8")],
    )

# 删除非特征列
non_feature_cols = ["event", "time_to_hit_hours"]
if "event_id" in df_train.columns:
    non_feature_cols.append("event_id")

X = df_train.drop(columns=non_feature_cols, errors="ignore")
y = prepare_y(df_train["event"], df_train["time_to_hit_hours"])

# 简单缺失值处理 + 标准化
X = X.fillna(X.mean())
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)

# 划分训练集/验证集，同时保留索引用于对齐
X_train, X_val, y_train, y_val, idx_train, idx_val = train_test_split(
    X_scaled, y, df_train.index, test_size=0.2, random_state=42
)

print("预处理与数据分割完成！")

预处理与数据分割完成！


# 3. 训练模型并输出预测
训练 Cox 模型，并生成 12/24/48/72 小时的生存概率，保存到 CSV。

In [8]:
# 方案 3：为训练集追加一个右删失伪样本，扩展随访到 72 小时
# 注意：这会影响模型参数，只用于让预测能覆盖 72 小时

# 构造伪样本特征（用训练集均值）
X_pseudo = pd.DataFrame([X_train.mean()], columns=X_train.columns)

y_pseudo = np.array([(False, 72.0001)], dtype=[("event", "?"), ("duration", "<f8")])

X_train_ext = pd.concat([X_train, X_pseudo], ignore_index=True)
y_train_ext = np.concatenate([y_train, y_pseudo])

cox_model = CoxPHSurvivalAnalysis(alpha=0.1)
print("开始训练 Cox 回归模型...")
cox_model.fit(X_train_ext, y_train_ext)

surv_funcs = cox_model.predict_survival_function(X_val)

# 直接使用固定时间点
raw_times = np.array([12, 24, 48, 72], dtype=float)

probabilities = np.asarray([[fn(t) for t in raw_times] for fn in surv_funcs])

# 组织输出文件
pred_cols = ["pred_12", "pred_24", "pred_48", "pred_72"]
preds_df = pd.DataFrame(probabilities, columns=pred_cols)

# 追加 risk_score 供 C-index 使用
preds_df["risk_score"] = cox_model.predict(X_val)

# 追加 event_id（如果存在）用于对齐
if "event_id" in df_train.columns:
    preds_df["event_id"] = df_train.loc[idx_val, "event_id"].to_numpy()
else:
    preds_df["event_id"] = idx_val

# 保存预测结果供评分笔记本使用
output_path = "data_clean/preds_baseline.csv"
preds_df.to_csv(output_path, index=False)
print(f"预测结果已保存到: {output_path}")

开始训练 Cox 回归模型...
预测结果已保存到: data_clean/preds_baseline.csv


# 4. 测试集预测（生成提交文件）
读取测试集并输出 submission.csv。

In [ ]:
# 读取测试集
X_test_raw = pd.read_csv("data_raw/test.csv")

# 保持与训练集相同的特征列
feature_cols = X.columns

# 若测试集中缺列，补齐
missing_cols = [c for c in feature_cols if c not in X_test_raw.columns]
for c in missing_cols:
    X_test_raw[c] = np.nan

X_test = X_test_raw[feature_cols]

# 与训练集一致的缺失值处理 + 标准化
X_test = X_test.fillna(X.mean())
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=feature_cols)

# 预测测试集生存函数
surv_funcs_test = cox_model.predict_survival_function(X_test_scaled)
raw_times = np.array([12, 24, 48, 72], dtype=float)

# 生成累积风险概率（事件发生概率 = 1 - 生存概率）
prob_test = 1 - np.asarray([[fn(t) for t in raw_times] for fn in surv_funcs_test])

# 1. 严格控制概率在 [0, 1] 范围内
prob_test = np.clip(prob_test, 0.0, 1.0)

# 2. 按行强制执行单调性（prob_12h <= prob_24h <= prob_48h <= prob_72h）
# 使用 np.maximum.accumulate 函数计算单调递增
prob_test = np.maximum.accumulate(prob_test, axis=1)

# 组织提交文件
submit = pd.DataFrame(prob_test, columns=["prob_12h", "prob_24h", "prob_48h", "prob_72h"])
if "event_id" in X_test_raw.columns:
    submit.insert(0, "event_id", X_test_raw["event_id"].to_numpy())

# 3. 提交验证（ID需跟测试集完全匹配）
assert submit.shape[0] == X_test_raw.shape[0], "行数不匹配！"
assert submit["event_id"].nunique() == submit.shape[0], "存在重复的 event_id！"
assert submit["event_id"].isna().sum() == 0, "存在缺失的 event_id！"

submit_path = "data_clean/Cox_Submission.csv"
submit.to_csv(submit_path, index=False)
print(f"提交文件已保存到: {submit_path}")
print("提交验证通过！")

提交文件已保存到: data_clean/Cox_Submission.csv
